# 📉 Telco Customer Churn Prediction (Classification)
**Target:** `Churn` — Yes / No

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 2. Load Dataset

In [ ]:
try:
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
except FileNotFoundError:
    print("CSV not found."); raise
print(f"Shape: {df.shape}")
df.head()

## 3. Identify Data Types

In [ ]:
print("Column data types:"); print(df.dtypes)
print(f"\nNumeric : {df.select_dtypes(include='number').columns.tolist()}")
print(f"Object  : {df.select_dtypes(include='object').columns.tolist()}")

## 4. Descriptive Statistics

In [ ]:
df.describe().round(2)

## 5. Handle Missing Values

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Missing values:"); print(df.isnull().sum()[df.isnull().sum()>0])
df.dropna(inplace=True)
print(f"Shape after drop: {df.shape} | NaNs: {df.isnull().sum().sum()}")

## 6. Handle Duplicates

In [ ]:
print(f"Duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Shape after dedup: {df.shape}")

## 7. Outlier Detection & Handling

In [ ]:
num_cols = ['tenure','MonthlyCharges','TotalCharges']
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax,col in zip(axes,num_cols):
    ax.boxplot(df[col],vert=False,patch_artist=True,boxprops=dict(facecolor='steelblue',alpha=0.6))
    ax.set_title(col,fontsize=9); ax.set_yticks([])
plt.suptitle('Boxplots',fontsize=12,y=1.02); plt.tight_layout(); plt.show()

before=len(df)
for col in num_cols:
    Q1,Q3=df[col].quantile([0.25,0.75]); IQR=Q3-Q1
    df=df[df[col].between(Q1-1.5*IQR,Q3+1.5*IQR)]
print(f"Removed: {before-len(df)} | Shape: {df.shape}")

## 8. Visualizations & Insights

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
counts=df['Churn'].value_counts()
axes[0].bar(counts.index,counts.values,color=['#2ecc71','#e74c3c'],edgecolor='white',width=0.4)
axes[0].set_title('Churn Distribution'); axes[0].set_ylabel('Count')
for i,(l,v) in enumerate(counts.items()): axes[0].text(i,v+10,str(v),ha='center',fontweight='bold')

df.boxplot(column='MonthlyCharges',by='Churn',ax=axes[1])
plt.sca(axes[1]); plt.title('Monthly Charges by Churn'); axes[1].set_xlabel('Churn')

df.boxplot(column='tenure',by='Churn',ax=axes[2])
plt.sca(axes[2]); plt.title('Tenure by Churn'); axes[2].set_xlabel('Churn')

plt.suptitle(''); plt.tight_layout(); plt.show()
print("""Insights:
1. ~73% No churn, ~27% Yes — imbalanced dataset.
2. Churned customers pay higher monthly charges.
3. Churned customers have shorter tenure — early-stage customers at higher risk.""")

## 9. Encode & Scale Features

In [ ]:
df.drop(columns=['customerID'],inplace=True)
le=LabelEncoder(); df['Churn']=le.fit_transform(df['Churn'])
cat_cols=df.select_dtypes(include='object').columns.tolist()
print(f"Encoding: {cat_cols}")
df_enc=pd.get_dummies(df,columns=cat_cols,drop_first=True)

X=df_enc.drop(columns=['Churn']); y=df_enc['Churn']
FEATURE_COLS=X.columns.tolist()
num_scale=['tenure','MonthlyCharges','TotalCharges']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
scaler=StandardScaler()
X_train[num_scale]=scaler.fit_transform(X_train[num_scale])
X_test[num_scale] =scaler.transform(X_test[num_scale])
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 10. Model Building

In [ ]:
models={
    'Logistic Regression': LogisticRegression(max_iter=1000,random_state=42,class_weight='balanced'),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5,random_state=42,class_weight='balanced'),
    'Random Forest'      : RandomForestClassifier(n_estimators=100,max_depth=6,random_state=42,class_weight='balanced')
}
results={}; preds={}; probs={}
for name,model in models.items():
    model.fit(X_train,y_train)
    yp=model.predict(X_test); ypr=model.predict_proba(X_test)[:,1]
    preds[name]=yp; probs[name]=ypr
    results[name]={'Accuracy':round(accuracy_score(y_test,yp),4),'ROC-AUC':round(roc_auc_score(y_test,ypr),4)}
    print(f"\n{'='*40}\n  {name}\n{'='*40}")
    print(classification_report(y_test,yp,target_names=['No Churn','Churn']))

## 11. Model Comparison

In [ ]:
res_df=pd.DataFrame(results).T.sort_values('ROC-AUC',ascending=False); print(res_df)
colors=['#3498db','#e67e22','#2ecc71']
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].bar(res_df.index,res_df['Accuracy'],color=colors,edgecolor='white',width=0.4)
axes[0].set_title('Accuracy',fontweight='bold'); axes[0].set_ylim(0,1)
axes[0].set_xticklabels(res_df.index,rotation=15,ha='right',fontsize=9)
axes[1].bar(res_df.index,res_df['ROC-AUC'],color=colors,edgecolor='white',width=0.4)
axes[1].set_title('ROC-AUC',fontweight='bold'); axes[1].set_ylim(0,1)
axes[1].set_xticklabels(res_df.index,rotation=15,ha='right',fontsize=9)
for (name,ypr),color in zip(probs.items(),colors):
    fpr,tpr,_=roc_curve(y_test,ypr)
    axes[2].plot(fpr,tpr,label=f"{name} ({results[name]['ROC-AUC']:.3f})",color=color,linewidth=2)
axes[2].plot([0,1],[0,1],'k--',linewidth=1); axes[2].set_title('ROC Curves')
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR'); axes[2].legend(fontsize=8)
plt.suptitle('Model Comparison',fontsize=13,y=1.02); plt.tight_layout(); plt.show()

---
## 🔮 12. Predict Churn for Your Own Customer
**Edit the values below and run the cell.**

In [ ]:
# ╔══════════════════════════════════════════╗
# ║   ✏️  CHANGE THESE VALUES TO YOUR INPUT  ║
# ╚══════════════════════════════════════════╝

tenure            = 12        # Months with company (0–72)
monthly_charges   = 70.0      # Monthly bill ($)
total_charges     = 840.0     # Total amount billed ($)

# Categorical inputs (type exactly as shown)
gender            = 'Male'            # 'Male' or 'Female'
senior_citizen    = 0                 # 0 = No, 1 = Yes
partner           = 'Yes'             # 'Yes' or 'No'
dependents        = 'No'              # 'Yes' or 'No'
phone_service     = 'Yes'             # 'Yes' or 'No'
multiple_lines    = 'No'              # 'Yes', 'No', 'No phone service'
internet_service  = 'Fiber optic'     # 'DSL', 'Fiber optic', 'No'
online_security   = 'No'              # 'Yes', 'No', 'No internet service'
online_backup     = 'No'              # 'Yes', 'No', 'No internet service'
device_protection = 'No'              # 'Yes', 'No', 'No internet service'
tech_support      = 'No'              # 'Yes', 'No', 'No internet service'
streaming_tv      = 'No'              # 'Yes', 'No', 'No internet service'
streaming_movies  = 'No'              # 'Yes', 'No', 'No internet service'
contract          = 'Month-to-month'  # 'Month-to-month', 'One year', 'Two year'
paperless_billing = 'Yes'             # 'Yes' or 'No'
payment_method    = 'Electronic check' # 'Electronic check','Mailed check','Bank transfer (automatic)','Credit card (automatic)'

# ── Build input row ───────────────────────
new_row = pd.DataFrame([{
    'tenure':tenure, 'MonthlyCharges':monthly_charges, 'TotalCharges':total_charges,
    'SeniorCitizen':senior_citizen,
    'gender':gender, 'Partner':partner, 'Dependents':dependents,
    'PhoneService':phone_service, 'MultipleLines':multiple_lines,
    'InternetService':internet_service, 'OnlineSecurity':online_security,
    'OnlineBackup':online_backup, 'DeviceProtection':device_protection,
    'TechSupport':tech_support, 'StreamingTV':streaming_tv,
    'StreamingMovies':streaming_movies, 'Contract':contract,
    'PaperlessBilling':paperless_billing, 'PaymentMethod':payment_method
}])

# Encode same way as training
new_enc = pd.get_dummies(new_row)
new_enc = new_enc.reindex(columns=FEATURE_COLS, fill_value=0)
new_enc[num_scale] = scaler.transform(new_enc[num_scale])

print("=" * 45)
print("      📉 CHURN PREDICTION RESULTS")
print("=" * 45)
print(f"  Tenure           : {tenure} months")
print(f"  Monthly Charges  : ${monthly_charges}")
print(f"  Contract         : {contract}")
print("-" * 45)
for name, model in models.items():
    pred = model.predict(new_enc)[0]
    prob = model.predict_proba(new_enc)[0][1]
    label = '🔴 Will Churn' if pred == 1 else '🟢 Will Stay'
    print(f"  {name:<22}: {label}  (prob: {prob:.2%})")
print("=" * 45)